# Position ladder — does primacy track parameters or training compute?

**Run this notebook once per model.** Set `MODEL_KEY` in cell [1] to one of
`llama2-7b`, `llama2-13b`, `llama31-8b`, `olmo3-7b`.

Loading two models in one runtime is the untested path that most likely killed
wide-catalog attempt 2 (76 min, no artifacts). One model per runtime, and results
persist after **every position block**, not at the end.

`llama2-7b` / `llama2-13b` are the **reproduction anchor**: they must return
`RECENCY_ONLY` / `U_SHAPE` or the ladder is uninterpretable and `evaluate()`
refuses to name a driver. Run the anchor rows first.

Adjudicator `position_ladder.py` embedded byte-identical (`51e9130e094dfcb4…`).
Scoring is mechanical UUID exact-match. No judge.

A100 assumed for all four rows so dtype is constant. Est. **~0.3 unit-hours per
7-8B row, ~0.5 for 13B**; `CONFIRMED_BUDGET` is per-row and also enforced
*during* the loop, not only at startup.

**Runtime:** system RAM is not the constraint — weights stream shard-by-shard to
GPU, so peak host RAM stays near one shard. **VRAM is the constraint.** On a
40 GB A100, Llama-2-13B (26 GB of bf16 weights, MHA so no GQA on the KV cache)
leaves little room: `BATCH = 4`, and cell [7] runs a warm-up probe at the worst-
case prompt length that fails in seconds rather than OOMing mid-run. Cell [5]
separately asserts nothing was offloaded to CPU or disk, since `device_map="auto"`
degrades silently instead of failing.


In [ ]:
# [1] preflight -- MODEL_KEY is the only thing you set
MODEL_KEY = "llama2-7b"          # llama2-7b | llama2-13b | llama31-8b | olmo3-7b
CONFIRMED_BUDGET_HOURS = 1.5     # per row; anomaly threshold, not a target

import torch, time
T0 = time.time()
assert torch.cuda.is_available(), "no GPU"
p = torch.cuda.get_device_properties(0)
GIB = p.total_memory / 1024**3
print(p.name, round(GIB, 1), "GiB")

# 13B fp16/bf16 needs ~26GB of weights alone.
if MODEL_KEY == "llama2-13b":
    assert GIB >= 38, f"13B anchor needs A100-40G+, got {GIB:.1f} GiB. Do NOT " \
                      "quantize: the confound would land on the anchor row."
else:
    assert GIB >= 20, f"need >=20 GiB, got {GIB:.1f}"

# bf16 where native. All four rows must share a dtype -- it is pinned into the
# output and the aggregator refuses to mix.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) \
        else torch.float16
print("dtype", DTYPE)

import transformers
assert transformers.__version__.startswith("5."), transformers.__version__

def elapsed_h():
    return (time.time() - T0) / 3600.0

def budget_check(where):
    """CONFIRMED_BUDGET only firing at startup provably cannot catch an overrun.
    Wide-catalog attempt 2 ran 76 minutes and persisted nothing."""
    e = elapsed_h()
    assert e <= CONFIRMED_BUDGET_HOURS, (
        f"OVERRUN at {where}: {e:.2f}h > {CONFIRMED_BUDGET_HOURS}h. "
        "Partial results are already persisted. Diagnose -- do not raise this "
        "to fit; a blowout here means the cost model is wrong.")
    return e


In [ ]:
# [2] pinned adjudicator + self-test ON THE BOX before any spend
import hashlib, pathlib
PL_SHA = "51e9130e094dfcb457881f0b9cf3c1f5108508ee28b0959c3f120b6c2fd91795"
PL_SRC = '"""Position-curve ladder: does primacy track PARAMETER COUNT or TRAINING COMPUTE?\n\nTHE QUESTION. Liu et al. (Lost in the Middle, TACL 2024) report Llama-2 at 7B is\nrecency-biased only, while 13B and 70B show the full U-shape with primacy. Arm G\nfound Llama-3.1-8B prefers catalog line 1 (primacy) while Olmo-3-7B prefers line 2\n(recency) -- two models at the SAME parameter count, opposite signs. If primacy\nwere set by parameter count, Llama-3.1-8B should look like Llama-2-7B. It does not.\n\nSo the ladder crosses the two candidate drivers:\n\n    Llama-2-7B-chat      7B    ~2T tokens    Liu et al. predict RECENCY_ONLY\n    Llama-2-13B-chat    13B    ~2T tokens    Liu et al. predict U_SHAPE\n    Llama-3.1-8B-Inst    8B   ~15T tokens    Arm G suggests primacy present\n    Olmo-3-7B-Inst       7B    ~6T tokens    Arm G suggests recency\n\nThe first two are the REPRODUCTION ANCHOR, not the result. If they do not\nreproduce Liu et al., this instrument does not measure what it claims and the\nother two rows are uninterpretable. That gate is in `evaluate()` and fires first.\n\nWHAT THIS IS NOT. It is not the Arm G line-position effect. That was 2 lines at\nconstant knowledge; this is 50 keys at constant task. A shared answer would be a\nconjecture, not a finding. Arm G\'s K=4 test went void twice and is still void.\n\nDESIGN, fixed here:\n  - Liu et al.\'s synthetic key-value retrieval. Chosen over their NQ-open task\n    because scoring is exact string match on a UUID -- no judge, no semantic\n    call. This repo has been burned by graders, not by models.\n  - N=50 pairs. BINDING CONSTRAINT: Llama-2 has a 4096-token window and every\n    model must see byte-identical prompts or position curves are not comparable.\n    50 pairs ~ 2.3k tokens leaves headroom for the chat template.\n  - 10 gold positions, evenly spaced, INCLUDING both endpoints -- endpoints carry\n    the entire primacy/recency signal.\n  - PAIRED: the same trial set (same UUIDs, same order) goes to every model, so a\n    model contrast is never confounded with the prompt draw.\n\nRun:  python3 position_ladder.py --self-test\n"""\nfrom __future__ import annotations\n\nimport json\nimport random\nimport re\nfrom typing import Any\n\nN_PAIRS = 50\nN_POSITIONS = 10\nTRIALS_PER_POSITION = 200\n\n# Ladder rows. `revision` is pinned at run time into the output record.\nLADDER = {\n    "llama2-7b":   dict(hf="meta-llama/Llama-2-7b-chat-hf",        params_b=7,  tokens_t=2.0,  role="anchor"),\n    "llama2-13b":  dict(hf="meta-llama/Llama-2-13b-chat-hf",       params_b=13, tokens_t=2.0,  role="anchor"),\n    "llama31-8b":  dict(hf="meta-llama/Llama-3.1-8B-Instruct",     params_b=8,  tokens_t=15.0, role="test"),\n    "olmo3-7b":    dict(hf="allenai/Olmo-3-7B-Instruct",           params_b=7,  tokens_t=6.0,  role="test"),\n}\n\n# Liu et al.\'s predictions for the anchor rows, fixed before any data exists.\nANCHOR_PREDICTION = {"llama2-7b": "RECENCY_ONLY", "llama2-13b": "U_SHAPE"}\n\nPROMPT = ("Extract the value corresponding to the specified key in the JSON "\n          "object below.\\n\\nJSON data:\\n{blob}\\n\\nKey: \\"{key}\\"\\n"\n          "Reply with the value only.")\n\nUUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")\n\n\ndef _uuid(rng: random.Random) -> str:\n    h = "%032x" % rng.getrandbits(128)\n    return f"{h[:8]}-{h[8:12]}-{h[12:16]}-{h[16:20]}-{h[20:]}"\n\n\ndef gold_positions(n_pairs: int = N_PAIRS, n_pos: int = N_POSITIONS) -> list[int]:\n    """Evenly spaced, endpoints included. Endpoints are the whole signal."""\n    return [round(i * (n_pairs - 1) / (n_pos - 1)) for i in range(n_pos)]\n\n\ndef build_trials(seed: int = 1101, n_pairs: int = N_PAIRS,\n                 trials_per_position: int = TRIALS_PER_POSITION) -> list[dict[str, Any]]:\n    """Key-value retrieval trials, gold key crossed over position.\n\n    The SAME returned list is served to every model in the ladder.\n    """\n    rng = random.Random(seed)\n    positions = gold_positions(n_pairs)\n    rows: list[dict[str, Any]] = []\n    for t in range(trials_per_position):\n        for pos in positions:\n            keys = [_uuid(rng) for _ in range(n_pairs)]\n            vals = [_uuid(rng) for _ in range(n_pairs)]\n            blob = json.dumps(dict(zip(keys, vals)), indent=2)\n            rows.append(dict(\n                trial=t, gold_position=pos, n_pairs=n_pairs,\n                gold_key=keys[pos], gold_value=vals[pos],\n                prompt=PROMPT.format(blob=blob, key=keys[pos]),\n            ))\n    return rows\n\n\ndef score_one(completion: str, gold_key: str, gold_value: str) -> dict[str, Any]:\n    """Mechanical. No judge, no fuzzy match.\n\n    The queried key is itself a UUID, so a model that echoes the key before\n    answering would be scored on the echo. Strip the key first, then take the\n    FIRST remaining UUID -- taking `gold_value in completion` instead would count\n    a model that dumps several candidates as correct.\n\n    `scorable` is the reachability precondition: a completion with no candidate\n    UUID at all is not a wrong retrieval, it is an unreadable trial, and a\n    position curve built on unreadable trials measures nothing.\n    """\n    cands = [u for u in UUID_RE.findall(completion) if u != gold_key]\n    if not cands:\n        return dict(scorable=False, correct=False, n_candidates=0, emitted=None)\n    return dict(scorable=True, correct=cands[0] == gold_value,\n                n_candidates=len(cands), emitted=cands[0])\n\n\ndef curve(scored: list[dict[str, Any]]) -> dict[int, float]:\n    """Accuracy by gold position, over SCORABLE trials only."""\n    import collections\n    num: dict[int, int] = collections.Counter()\n    den: dict[int, int] = collections.Counter()\n    for r in scored:\n        if not r["scorable"]:\n            continue\n        den[r["gold_position"]] += 1\n        num[r["gold_position"]] += int(r["correct"])\n    return {p: num[p] / den[p] for p in sorted(den) if den[p]}\n\n\n# --------------------------------------------------------------------------\n# Adjudicator. Committed before any GPU spend; the verdict is whatever this\n# returns. Void branches precede shape branches -- an unreachable task cannot\n# have a shape.\n\nVOID_VERDICTS = frozenset({"V_UNSCORABLE", "V_TASK_UNREACHABLE", "V_UNREADABLE"})\n\nBARS = dict(index=0.10, scorable=0.75, ceiling=0.20)\nSENS = dict(index=(0.05, 0.15), scorable=(0.60, 0.85), ceiling=(0.10, 0.30))\n\n\ndef classify(c: dict[int, float], bars: dict = BARS) -> dict[str, Any]:\n    """Curve shape from endpoint lift over the interior MEAN.\n\n    NOT the interior minimum. The minimum of 8 noisy position estimates is a\n    downward-biased statistic -- at n=200/position it sits roughly 1.5 SE below\n    the true interior level, which inflates BOTH indices by ~5pp against a 10pp\n    bar and pushes genuinely flat curves into U_SHAPE. The mean is unbiased and\n    pools 8x the data, so its own error is negligible beside the endpoints\'.\n\n    The reason a min baseline looked attractive -- that a mean baseline lets a\n    monotone rise read as recency -- is not a defect: Liu et al.\'s 7B curve IS\n    essentially a monotone rise toward the end, and RECENCY_ONLY is the correct\n    reading of it. A monotone rise also scores NEGATIVE primacy here, so it can\n    never be mistaken for a U.\n    """\n    if len(c) < 3:\n        return dict(verdict="V_UNREADABLE", reasons=[f"only {len(c)} positions"])\n    ps = sorted(c)\n    first, last = c[ps[0]], c[ps[-1]]\n    interior = [c[p] for p in ps[1:-1]]\n    floor = sum(interior) / len(interior)\n    prim, rec = first - floor, last - floor\n    if prim >= bars["index"] and rec >= bars["index"]:\n        v = "U_SHAPE"\n    elif rec >= bars["index"]:\n        v = "RECENCY_ONLY"\n    elif prim >= bars["index"]:\n        v = "PRIMACY_ONLY"\n    else:\n        v = "FLAT"\n    return dict(verdict=v, primacy_index=round(prim, 4), recency_index=round(rec, 4),\n                interior_mean=round(floor, 4), interior_min=round(min(interior), 4),\n                first=round(first, 4), last=round(last, 4),\n                curve={p: round(c[p], 4) for p in ps})\n\n\ndef evaluate(m: dict[str, Any], bars: dict = BARS) -> dict[str, Any]:\n    """m = {model_key: {curve: {pos: acc}, scorable_frac: float}}\n\n    Anchor gate first: if Llama-2 7B/13B do not reproduce Liu et al., the\n    instrument is unvalidated and the test rows carry no interpretation.\n    """\n    per: dict[str, Any] = {}\n    for key, d in m.items():\n        sf = d.get("scorable_frac", 1.0)\n        c = d.get("curve") or {}\n        if sf < bars["scorable"]:\n            per[key] = dict(verdict="V_UNSCORABLE",\n                            reasons=[f"scorable {sf:.3f} < {bars[\'scorable\']}"])\n            continue\n        if not c or max(c.values()) < bars["ceiling"]:\n            top = max(c.values()) if c else 0.0\n            per[key] = dict(verdict="V_TASK_UNREACHABLE",\n                            reasons=[f"max acc {top:.3f} < {bars[\'ceiling\']}; "\n                                     f"model cannot do the task at N={N_PAIRS}"])\n            continue\n        per[key] = classify(c, bars)\n\n    anchors = {k: per[k]["verdict"] for k in ANCHOR_PREDICTION if k in per}\n    missing = [k for k in ANCHOR_PREDICTION if k not in per]\n    mismatch = {k: (v, ANCHOR_PREDICTION[k]) for k, v in anchors.items()\n                if v != ANCHOR_PREDICTION[k]}\n    if missing:\n        gate = dict(status="INCOMPLETE", reasons=[f"anchor rows absent: {missing}"])\n    elif mismatch:\n        gate = dict(status="ANCHOR_FAILED", reasons=[\n            f"{k}: got {got}, Liu et al. predict {want}"\n            for k, (got, want) in mismatch.items()])\n    else:\n        gate = dict(status="ANCHOR_OK", reasons=["Liu et al. reproduced at 7B and 13B"])\n\n    out = dict(anchor_gate=gate, per_model=per)\n    if gate["status"] != "ANCHOR_OK":\n        out["driver"] = None\n        out["reasons"] = ["instrument unvalidated; test rows not interpreted"]\n        return out\n\n    # The actual contrast: same parameter band, different training compute.\n    #\n    # A VOID row is not a row without primacy. Coercing `V_UNSCORABLE` to\n    # `has_primacy=False` would let three unreadable models return\n    # PARAMETER_COUNT -- absence of measurement read as measurement of absence,\n    # which is the defect class this repo keeps finding. Void rows are dropped\n    # and their absence blocks the driver claim.\n    small = {k: per[k]["verdict"] for k in ("llama2-7b", "llama31-8b", "olmo3-7b")\n             if k in per}\n    void = {k: v for k, v in small.items() if v in VOID_VERDICTS}\n    has_primacy = {k: v in ("U_SHAPE", "PRIMACY_ONLY")\n                   for k, v in small.items() if v not in VOID_VERDICTS}\n    if len(has_primacy) < 3:\n        out["driver"] = None\n        out["reasons"] = ["need all three readable 7-8B rows to separate the drivers"] + (\n            [f"void: {void}"] if void else\n            [f"missing: {sorted({\'llama2-7b\',\'llama31-8b\',\'olmo3-7b\'} - set(small))}"])\n    elif has_primacy.get("llama2-7b") is False and any(\n            has_primacy.get(k) for k in ("llama31-8b", "olmo3-7b")):\n        out["driver"] = "TRAINING_COMPUTE"\n        out["reasons"] = ["primacy present at 7-8B for a later-generation model "\n                          "and absent for Llama-2-7B; parameter count held"]\n    elif not any(has_primacy.values()):\n        out["driver"] = "PARAMETER_COUNT"\n        out["reasons"] = ["no 7-8B model shows primacy regardless of training; "\n                          "consistent with a parameter-count threshold"]\n    else:\n        out["driver"] = "UNRESOLVED"\n        out["reasons"] = [f"primacy pattern does not separate the drivers: {has_primacy}"]\n    return out\n\n\ndef sensitivity(m: dict[str, Any]) -> dict[str, Any]:\n    base = evaluate(m)\n    bk = (base["anchor_gate"]["status"], base.get("driver"))\n    grid = {}\n    for key, vals in SENS.items():\n        for v in vals:\n            b = dict(BARS); b[key] = v\n            r = evaluate(m, b)\n            grid[f"{key}={v}"] = (r["anchor_gate"]["status"], r.get("driver"))\n    return dict(base=bk, grid=grid, fragile=any(x != bk for x in grid.values()))\n\n\n# N_PAIRS cannot be fixed from here: UUID text tokenizes at a rate that depends\n# on the tokenizer, and Llama-2\'s 4096 window is the binding constraint for the\n# whole ladder. So the RULE is fixed instead, before any data -- largest\n# candidate that fits, chosen once at Phase 0 with the real Llama-2 tokenizer and\n# pinned into every output record. N is a context-window fact, never a knob to\n# turn after seeing a curve.\nN_CANDIDATES = (50, 40, 30, 20)\nCTX_HEADROOM = 256\n\n\ndef select_n_pairs(tokenizer, max_ctx: int = 4096,\n                   candidates: tuple[int, ...] = N_CANDIDATES) -> tuple[int, int]:\n    """(n_pairs, worst_case_tokens). Phase 0. Call before ANY generation.\n\n    A prompt that silently truncates loses the FRONT of the JSON -- it would\n    delete exactly the primacy signal this study measures and return a clean\n    recency curve that looks like a result.\n    """\n    budget = max_ctx - CTX_HEADROOM\n    for n in candidates:\n        rows = build_trials(trials_per_position=2, n_pairs=n)\n        worst = max(len(tokenizer(r["prompt"])["input_ids"]) for r in rows)\n        if worst < budget:\n            return n, worst\n    raise AssertionError(\n        f"no candidate in {candidates} fits {budget} tokens; the ladder cannot "\n        f"hold prompts constant across models and must be redesigned")\n\n\ndef assert_fits(tokenizer, n_pairs: int, max_ctx: int = 4096) -> int:\n    """Re-assert the pinned N still fits, on EVERY model\'s tokenizer."""\n    rows = build_trials(trials_per_position=2, n_pairs=n_pairs)\n    worst = max(len(tokenizer(r["prompt"])["input_ids"]) for r in rows)\n    assert worst < max_ctx - CTX_HEADROOM, (\n        f"prompt is {worst} tokens against a {max_ctx} window; truncation would "\n        f"remove the early keys and manufacture a recency result")\n    return worst\n\n\ndef _selftest() -> None:\n    rows = build_trials(trials_per_position=2)\n    assert len(rows) == 2 * N_POSITIONS\n    assert gold_positions()[0] == 0 and gold_positions()[-1] == N_PAIRS - 1\n    for r in rows:\n        assert r["gold_key"] in r["prompt"] and r["gold_value"] in r["prompt"]\n        assert r["prompt"].count(r["gold_key"]) == 2  # once in JSON, once in Key:\n    # paired design: same seed -> byte-identical prompts\n    assert [r["prompt"] for r in build_trials(trials_per_position=2)] == \\\n           [r["prompt"] for r in rows], "trial set is not reproducible"\n    # every model must get the SAME prompts, so no model arg exists in build_trials\n    assert "model" not in build_trials.__code__.co_varnames\n\n    # scorer\n    k, v = rows[0]["gold_key"], rows[0]["gold_value"]\n    assert score_one(f"The value is {v}", k, v)["correct"] is True\n    assert score_one(f\'"{k}": "{v}"\', k, v)["correct"] is True, "key echo must be stripped"\n    other = _uuid(random.Random(9))\n    assert score_one(f"{other} but maybe {v}", k, v)["correct"] is False, \\\n        "first non-key candidate wins; a dump of guesses is not a retrieval"\n    s = score_one("I cannot find that key.", k, v)\n    assert s["scorable"] is False and s["correct"] is False\n    assert score_one(f"{v} {other}", k, v)["n_candidates"] == 2\n\n    # curve + classify\n    def mk(d):\n        return {p: a for p, a in d.items()}\n    u = mk({0: .9, 10: .4, 20: .35, 30: .4, 40: .5, 49: .85})\n    assert classify(u)["verdict"] == "U_SHAPE"\n    r_only = mk({0: .40, 10: .38, 20: .35, 30: .40, 40: .55, 49: .80})\n    assert classify(r_only)["verdict"] == "RECENCY_ONLY"\n    p_only = mk({0: .80, 10: .38, 20: .35, 30: .40, 40: .42, 49: .40})\n    assert classify(p_only)["verdict"] == "PRIMACY_ONLY"\n    assert classify(mk({0: .5, 10: .48, 20: .47, 30: .5, 49: .52}))["verdict"] == "FLAT"\n    # a monotone rise must NOT read as recency-with-primacy, and must score\n    # NEGATIVE primacy so it can never drift into U_SHAPE\n    mono = classify(mk({0: .2, 10: .3, 20: .4, 30: .5, 40: .6, 49: .7}))\n    assert mono["verdict"] == "RECENCY_ONLY" and mono["primacy_index"] < 0, mono\n\n    # REGRESSION for the min-baseline bias: a flat curve whose interior has one\n    # unlucky low position must stay FLAT. Against interior MIN both indices\n    # would clear 0.10 and this would have read as a U-shape.\n    noisy_flat = mk({0: .50, 10: .49, 20: .38, 30: .50, 40: .51, 49: .52})\n    r = classify(noisy_flat)\n    assert r["verdict"] == "FLAT", r\n    assert r["first"] - r["interior_min"] > BARS["index"], \\\n        "fixture no longer exercises the bias it was written for"\n\n    def row(c, sf=1.0):\n        return dict(curve=c, scorable_frac=sf)\n\n    # anchor gate fires BEFORE any driver claim\n    bad = evaluate({"llama2-7b": row(u), "llama2-13b": row(u)})\n    assert bad["anchor_gate"]["status"] == "ANCHOR_FAILED"\n    assert bad["driver"] is None, "driver claimed on an unvalidated instrument"\n\n    ok = {"llama2-7b": row(r_only), "llama2-13b": row(u),\n          "llama31-8b": row(p_only), "olmo3-7b": row(r_only)}\n    e = evaluate(ok)\n    assert e["anchor_gate"]["status"] == "ANCHOR_OK"\n    assert e["driver"] == "TRAINING_COMPUTE", e\n\n    # all three small rows recency -> parameter count\n    e2 = evaluate({"llama2-7b": row(r_only), "llama2-13b": row(u),\n                   "llama31-8b": row(r_only), "olmo3-7b": row(r_only)})\n    assert e2["driver"] == "PARAMETER_COUNT", e2\n\n    # void precedes shape, on both void paths -- and a void row must BLOCK the\n    # driver claim rather than counting as "no primacy here"\n    e3 = evaluate({**ok, "olmo3-7b": row(u, sf=0.30)})\n    assert e3["per_model"]["olmo3-7b"]["verdict"] == "V_UNSCORABLE"\n    assert e3["driver"] is None, e3\n    assert any("olmo3-7b" in r for r in e3["reasons"]), e3["reasons"]\n    e4 = evaluate({**ok, "olmo3-7b": row(mk({0: .05, 10: .03, 20: .02, 49: .04}))})\n    assert e4["per_model"]["olmo3-7b"]["verdict"] == "V_TASK_UNREACHABLE"\n    assert e4["driver"] is None, e4\n    # THE REGRESSION THAT MOTIVATED VOID_VERDICTS: three unreadable models must\n    # never return PARAMETER_COUNT ("no model showed primacy" from no data)\n    allvoid = evaluate({"llama2-7b": row(r_only), "llama2-13b": row(u),\n                        "llama31-8b": row(p_only, sf=0.1), "olmo3-7b": row(u, sf=0.1)})\n    assert allvoid["driver"] is None, \\\n        f"absence of measurement read as measurement of absence: {allvoid}"\n\n    # incomplete ladder cannot yield a driver\n    assert evaluate({"llama2-7b": row(r_only)})["driver"] is None\n\n    # N-selection rule, against a stub tokenizer with a known chars-per-token\n    class Stub:\n        def __init__(self, cpt): self.cpt = cpt\n        def __call__(self, s): return {"input_ids": [0] * (len(s) // self.cpt)}\n    n, w = select_n_pairs(Stub(3))          # roomy -> largest candidate\n    assert n == 50 and w < 4096 - CTX_HEADROOM, (n, w)\n    n, w = select_n_pairs(Stub(1))          # 1 char/token -> must step down\n    assert n < 50 and w < 4096 - CTX_HEADROOM, (n, w)\n    assert build_trials(trials_per_position=1, n_pairs=n)[0]["n_pairs"] == n\n    assert gold_positions(n)[-1] == n - 1, "positions must track the chosen N"\n    try:\n        select_n_pairs(Stub(1), max_ctx=200)\n        raise SystemExit("select_n_pairs accepted an impossible budget")\n    except AssertionError:\n        pass\n\n    s = sensitivity(ok)\n    assert s["base"] == ("ANCHOR_OK", "TRAINING_COMPUTE")\n    print(f"self-test OK: {len(rows)} trial rows, scorer key-echo + multi-candidate, "\n          f"5 shape worlds, anchor gate, 2 void paths, driver logic")\n    print(f"  positions: {gold_positions()}")\n    print(f"  full run: {N_POSITIONS * TRIALS_PER_POSITION} prompts/model "\n          f"x {len(LADDER)} models = {N_POSITIONS * TRIALS_PER_POSITION * len(LADDER)}")\n    print(f"  sensitivity fragile: {s[\'fragile\']}")\n\n\nif __name__ == "__main__":\n    import sys\n    if "--self-test" in sys.argv: _selftest()\n    else: print(__doc__)\n'
assert hashlib.sha256(PL_SRC.encode()).hexdigest() == PL_SHA, "artifact drift"
pathlib.Path("position_ladder.py").write_text(PL_SRC)

import position_ladder as PL
PL._selftest()
print("\nadjudicator self-test passed on this box; sha", PL_SHA[:16])


In [ ]:
# [3] persistence -- private HF dataset repo, keyed by config hash
# Drive's OAuth popup is unreachable in a hands-off run (verified). HF_TOKEN is
# a Colab secret, silent on fresh VMs, and uploads are atomic per file.
import os, io, json, hashlib
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

HF_TOKEN = userdata.get("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
WHO = api.whoami()["name"]
DS = f"{WHO}/phi-map-position-ladder"
create_repo(DS, repo_type="dataset", private=True, exist_ok=True, token=HF_TOKEN)
print("persisting to", DS)

WORK = "/content/pl"; os.makedirs(WORK, exist_ok=True)

def persist(name, blob: bytes):
    open(os.path.join(WORK, name), "wb").write(blob)
    api.upload_file(path_or_fileobj=io.BytesIO(blob), path_in_repo=name,
                    repo_id=DS, repo_type="dataset", token=HF_TOKEN)

def fetch(name):
    from huggingface_hub import hf_hub_download
    try:
        p = hf_hub_download(DS, name, repo_type="dataset", token=HF_TOKEN)
        return json.load(open(p))
    except Exception:
        return None


In [ ]:
# [4] PHASE 0 -- access gate for ALL FOUR repos, then pin N
# All four are checked even though this runtime runs one, because a 401 on a
# later row after three rows are already paid for is the expensive failure.
from huggingface_hub import model_info
from transformers import AutoTokenizer

# model_info() is NOT a sufficient probe: repo metadata and SHA are PUBLIC for
# gated repos, so it returns 200 for a repo you cannot read. This gate printed
# four "ok"s and then 403'd on the very next line -- the gate checked what it
# could check instead of what mattered, which is the failure it exists to
# prevent. Resolve an actual FILE, which is the operation that is gated.
from huggingface_hub import hf_hub_download

bad = []
for k, spec in PL.LADDER.items():
    try:
        hf_hub_download(spec["hf"], "config.json", token=HF_TOKEN)   # ~1 KB
        sha = model_info(spec["hf"], token=HF_TOKEN).sha
        print(f"  ok    {k:12s} {spec['hf']}  @{sha[:12]}")
    except Exception as e:
        bad.append((k, spec["hf"], type(e).__name__))
        print(f"  FAIL  {k:12s} {spec['hf']}  {type(e).__name__}: "
              f"{str(e).splitlines()[0][:90]}")
assert not bad, (
    f"cannot READ {[b[0] for b in bad]}. Request access on huggingface.co under "
    f"the same account as HF_TOKEN: " + ", ".join(f"https://huggingface.co/{b[1]}"
                                                   for b in bad) +
    ". Meta gates by approval, not by clickthrough, so this can take a while. "
    "Failing here costs nothing; failing at load time costs the rows already run.")

# N is a context-window fact, fixed once by Llama-2's 4096 window -- the binding
# constraint for the WHOLE ladder -- and identical for every row.
tok_l2 = AutoTokenizer.from_pretrained(PL.LADDER["llama2-7b"]["hf"], token=HF_TOKEN)
N_PAIRS, WORST = PL.select_n_pairs(tok_l2)
print(f"\nN_PAIRS = {N_PAIRS} (worst-case {WORST} tokens vs 4096 window)")
assert N_PAIRS in PL.N_CANDIDATES


In [ ]:
# [5] load THIS model, pin revision, re-assert the prompt fits ITS tokenizer
from transformers import AutoModelForCausalLM
SPEC = PL.LADDER[MODEL_KEY]
REPO = SPEC["hf"]
REV = model_info(REPO, token=HF_TOKEN).sha          # pinned into the output
print(MODEL_KEY, REPO, "@", REV[:12])

tok = AutoTokenizer.from_pretrained(REPO, revision=REV, token=HF_TOKEN)
worst_here = PL.assert_fits(tok, N_PAIRS,
                            max_ctx=4096 if MODEL_KEY.startswith("llama2") else 8192)
print("worst-case prompt on this tokenizer:", worst_here, "tokens")

model = AutoModelForCausalLM.from_pretrained(
    REPO, revision=REV, torch_dtype=DTYPE, device_map="auto", token=HF_TOKEN)
model.eval()

# device_map="auto" does NOT fail when VRAM is short -- it offloads layers to
# CPU and then to disk, and the run merely crawls. That is indistinguishable
# from a hang and is how wide-catalog attempt 2 burned 76 minutes and persisted
# nothing. Fail here instead.
dmap = getattr(model, "hf_device_map", {}) or {}
offloaded = {k: str(v) for k, v in dmap.items()
             if str(v) in ("cpu", "disk") or "cpu" in str(v) or "disk" in str(v)}
assert not offloaded, (
    f"{len(offloaded)} module(s) offloaded off-GPU: {list(offloaded)[:5]}. "
    f"Generation would crawl rather than fail. Use a larger GPU -- do not "
    f"quantize to fit, especially not on an anchor row.")
print(f"device map: {len(dmap)} modules, all on GPU"
      if dmap else "device map: single device")
print("VRAM after load: %.1f GiB" % (torch.cuda.memory_allocated() / 1024**3))

tok.padding_side = "left"                 # LEFT-pad: this is a GENERATION pass
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("padding_side", tok.padding_side, "| pad", tok.pad_token_id)


In [ ]:
# [6] SERVED-PROMPT GATE -- print what actually reaches the model
# The one cheap countermeasure that would have caught the burned wide-catalog
# run: every notebook that worked printed the SERVED prompt; the one that failed
# gated the manifest and never checked what the model was asked to do.
trials = PL.build_trials(n_pairs=N_PAIRS)
print(f"{len(trials)} trials = {PL.N_POSITIONS} positions x {PL.TRIALS_PER_POSITION}")

def served(row):
    msgs = [{"role": "user", "content": row["prompt"]}]
    out = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    # transformers v5 can return a dict/BatchEncoding even for tokenize=False
    if not isinstance(out, str):
        out = out[0] if isinstance(out, (list, tuple)) else str(out)
    return out

ex = served(trials[0])
print("=" * 78)
print(ex[:600], "\n   ...[middle elided]...\n", ex[-400:])
print("=" * 78)
assert trials[0]["gold_key"] in ex, "gold key absent from the served prompt"
assert trials[0]["gold_value"] in ex, "gold value absent from the served prompt"
assert "Extract the value" in ex, "served prompt does not ask for a retrieval"
assert ex.count(trials[0]["gold_key"]) == 2, "key must appear in JSON and in query"
print("served-prompt gate OK")

CONFIG = dict(model_key=MODEL_KEY, repo=REPO, revision=REV, dtype=str(DTYPE),
              n_pairs=N_PAIRS, seed=1101, trials_per_position=PL.TRIALS_PER_POSITION,
              positions=PL.gold_positions(N_PAIRS), pl_sha=PL_SHA)
CFG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]
print("config hash", CFG_HASH)


In [ ]:
# [7] generate -- greedy, checkpointed after EVERY position block
import collections
# BATCH is a THROUGHPUT knob, not a design parameter: greedy decoding with
# left-padding means batch composition does not change any row's answer. It is
# held constant across models anyway so the rows stay strictly comparable.
#
# It is 4, not 8, because Llama-2-13B is MHA (40 layers x 40 heads x 128, no
# GQA) and its KV cache at batch 8 x ~2600 tokens is ~17 GB on top of 26 GB of
# weights -- over a 40 GB card. The warm-up probe below settles it empirically
# instead of trusting that arithmetic.
BATCH, MAXNEW = 4, 64
CKPT = f"gen_{MODEL_KEY}_{CFG_HASH}.json"

done = fetch(CKPT) or {}
if done:
    print(f"resuming: {len(done)} position blocks already persisted")

# WARM-UP PROBE: real batch, real worst-case prompt length, measured peak VRAM.
# Fails in seconds instead of OOMing 300 batches in. Backs off automatically:
# Llama-2-13B at batch 4 lands ~38 GiB on a 40 GiB card and would otherwise
# force a manual re-run.
#
# BATCH is a throughput knob, NOT a design parameter -- greedy decoding with
# left-padding makes each sequence's forward pass independent, so batch size
# cannot change which UUID a row retrieves. It is therefore allowed to vary per
# model, is recorded in CONFIG, and is deliberately NOT one of the fields the
# aggregator's comparability guard checks. N, dtype, seed and trial count are
# design parameters and are held fixed.
_probe_rows = sorted(trials, key=lambda r: -len(r["prompt"]))
while True:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    try:
        _enc = tok([served(r) for r in _probe_rows[:BATCH]], return_tensors="pt",
                   padding=True, add_special_tokens=False).to(model.device)
        # Silent truncation would delete the FRONT of the JSON and hand back a
        # clean recency curve that looks like a result. Assert, do not hope.
        _ctx = 4096 if MODEL_KEY.startswith("llama2") else 8192
        assert _enc["input_ids"].shape[1] + MAXNEW <= _ctx, (
            f"worst-case prompt {_enc['input_ids'].shape[1]} + {MAXNEW} new "
            f"exceeds the {_ctx} window; N was mis-selected at Phase 0")
        with torch.no_grad():
            model.generate(**_enc, max_new_tokens=MAXNEW, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        PEAK = torch.cuda.max_memory_allocated() / 1024**3
        ok = PEAK < 0.90 * GIB
    except torch.cuda.OutOfMemoryError:
        PEAK, ok = float("inf"), False
    print(f"warm-up: batch {BATCH} len {_probe_rows[0] and _enc['input_ids'].shape[1]} "
          f"peak {PEAK:.1f} / {GIB:.1f} GiB ({100 * PEAK / GIB:.0f}%) "
          f"{'ok' if ok else 'too high, halving'}")
    del _enc
    if ok:
        break
    BATCH //= 2
    assert BATCH >= 1, (
        f"cannot fit even batch 1 in {GIB:.1f} GiB. Use a larger GPU. Do NOT "
        f"quantize -- especially not on an anchor row -- and do not shorten the "
        f"prompt, since N is fixed by the ladder.")
torch.cuda.empty_cache()
# Recorded separately, NOT folded into CONFIG: CFG_HASH is already computed and
# keys the resume checkpoint, so a run that backs off to a smaller batch must
# still resume the same file rather than starting over under a new key.
BATCH_USED, PEAK_GIB = BATCH, round(PEAK, 2)

by_pos = collections.defaultdict(list)
for r in trials:
    by_pos[r["gold_position"]].append(r)

for pos in sorted(by_pos):
    if str(pos) in done:
        print(f"  pos {pos:3d} cached"); continue
    rows, outs = by_pos[pos], []
    for i in range(0, len(rows), BATCH):
        batch = [served(r) for r in rows[i:i + BATCH]]
        enc = tok(batch, return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=MAXNEW, do_sample=False,
                               pad_token_id=tok.pad_token_id)
        new = g[:, enc["input_ids"].shape[1]:]
        outs += [tok.decode(s, skip_special_tokens=True) for s in new]
        budget_check(f"pos {pos} batch {i}")
    done[str(pos)] = [dict(gold_key=r["gold_key"], gold_value=r["gold_value"],
                           trial=r["trial"], text=t) for r, t in zip(rows, outs)]
    persist(CKPT, json.dumps(done).encode())     # after EVERY block
    print(f"  pos {pos:3d} done  {len(outs)} gens  {elapsed_h():.2f}h  persisted")
print("generation complete", f"{elapsed_h():.2f}h")


In [ ]:
# [8] score, curve, per-model verdict
scored = []
for pos, recs in done.items():
    for r in recs:
        s = PL.score_one(r["text"], r["gold_key"], r["gold_value"])
        s["gold_position"] = int(pos)
        scored.append(s)

curve = PL.curve(scored)
scorable_frac = sum(s["scorable"] for s in scored) / len(scored)
row_result = dict(curve={str(k): v for k, v in curve.items()},
                  scorable_frac=scorable_frac, n=len(scored))
verdict = PL.classify(curve) if scorable_frac >= PL.BARS["scorable"] else \
          dict(verdict="V_UNSCORABLE", reasons=[f"scorable {scorable_frac:.3f}"])

print(f"scorable {scorable_frac:.4f}  n={len(scored)}")
for p in sorted(curve):
    print(f"  pos {p:3d}  acc {curve[p]:.4f}")
print("\nverdict:", json.dumps(verdict, indent=1))
if MODEL_KEY in PL.ANCHOR_PREDICTION:
    want = PL.ANCHOR_PREDICTION[MODEL_KEY]
    print(f"\nANCHOR ROW: Liu et al. predict {want}, got {verdict['verdict']} "
          f"-> {'MATCH' if verdict['verdict'] == want else 'MISMATCH'}")


In [ ]:
# [9] hand verification + persist the row
print("=" * 78, "\nHAND VERIFICATION: 3 generations per outcome\n", "=" * 78)
buckets = {"correct": [], "wrong": [], "unscorable": []}
for pos, recs in done.items():
    for r in recs:
        s = PL.score_one(r["text"], r["gold_key"], r["gold_value"])
        b = "unscorable" if not s["scorable"] else ("correct" if s["correct"] else "wrong")
        if len(buckets[b]) < 3:
            buckets[b].append((pos, r, s))
for b, items in buckets.items():
    for pos, r, s in items:
        print(f"\n--- pos {pos} scored {b}")
        print("    gold :", r["gold_value"])
        print("    text :", repr(r["text"][:200]))

summary = dict(protocol="POSITION_LADDER_V1", config=CONFIG, cfg_hash=CFG_HASH,
               result=row_result, verdict=verdict,
               anchor_prediction=PL.ANCHOR_PREDICTION.get(MODEL_KEY),
               batch=BATCH_USED, peak_gib=PEAK_GIB, max_new_tokens=MAXNEW,
               wall_h=round(elapsed_h(), 3), gpu=p.name)
blob = json.dumps(summary, indent=1).encode()
persist(f"row_{MODEL_KEY}_{CFG_HASH}.json", blob)
print("\nsha256:", hashlib.sha256(blob).hexdigest()[:16])
print("=== ROW SUMMARY BEGIN ===")
print(json.dumps(summary, indent=1))
print("=== ROW SUMMARY END ===")
print("\nNext: re-run with the next MODEL_KEY on a FRESH runtime.")
print("When all four rows exist: python3 position_ladder_aggregate.py row_*.json")
